<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 6 — Fine-tuning de BERT</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Un backbone, tres cabezas — con datasets reales y demo sobre su corpus</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Entorno: Google Colab con GPU T4 (free tier).** Entorno de ejecución → Cambiar tipo → **GPU**. 
Los tres modelos (BETO 110M, Sentence-BERT) caben de sobra en los 15 GB de la T4 usando `fp16`. 
Esta versión entrena con **datasets reales de HuggingFace** (no con el corpus de juguete): `tweet_sentiment_multilingual` para clasificación y `conll2002` para NER. El corpus chiapaneco se usa solo para la **inferencia final** de cada parte. Las celdas de *liberar memoria* permiten correr A → B → C en una sola sesión sin saturar la VRAM.


## Objetivo

Tres partes, el mismo BERT preentrenado en español como base. **A)** Afinar un encoder de oraciones (Sentence-BERT) con sus pares del Lab 3. **B)** Clasificar **sentimiento** con un dataset real en español. **C)** **NER** con CoNLL-2002. Cada parte cierra **usando el modelo afinado** sobre el corpus chiapaneco, y libera memoria antes de la siguiente.


## 0 · Setup, GPU y utilidades

In [12]:
# !pip install -q transformers sentence-transformers datasets seqeval accelerate

import gc
import json
import math
import re
import unicodedata
from collections import Counter
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — activen el runtime de GPU')

def liberar_memoria():
    """Libera RAM/VRAM de forma agresiva entre bloques de entrenamiento."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'VRAM en uso: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Cargar el corpus del Lab 1
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
with open('corpus_crudo.json', encoding='utf-8') as fh:
    crudo = {d['id']: d['texto'] for d in json.load(fh)}

ids = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(f'{len(corpus)} documentos cargados con éxito para las fases de inferencia.')

GPU: NO GPU — activen el runtime de GPU
14 documentos cargados con éxito para las fases de inferencia.


In [13]:
# El corpus chiapaneco (del Lab 1) se usa SOLO para la inferencia final de cada parte.
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
with open('corpus_crudo.json', encoding='utf-8') as fh:
    crudo = {d['id']: d['texto'] for d in json.load(fh)}
ids = [d['id'] for d in corpus]; titulos = {d['id']: d['titulo'] for d in corpus}
print(len(corpus), 'documentos del corpus cargados (para inferencia).')

14 documentos del corpus cargados (para inferencia).


---
## Parte A · Embeddings con Sentence-BERT (datos: sus qrels del Lab 3)

> **Por qué aquí sí usamos el corpus.** El fine-tuning de embeddings necesita pares *de dominio* (consulta ↔ documento relevante). Esos pares salen de **sus** qrels del Lab 3: es el cierre del arco de la unidad, donde su juicio de relevancia se vuelve señal de entrenamiento. **Advertencia metodológica:** con ~5 consultas esto es una *demostración del método*, no un experimento — la mejora puede ser chica o ruidosa. Lo evaluable es el pipeline correcto, no el número.


**A.1** Línea base: carguen un Sentence-BERT en español y midan su buscador **sin afinar** con su nDCG del Lab 3.

In [14]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# Juicios de relevancia heredados del Lab 3 / Lab 5
qrels = {
    'problemas de agua potable': {'d13': 3, 'd02': 1, 'd01': 1},
    'sequia y cultivos de maiz': {'d04': 3, 'd02': 2},
    'crisis hidrica': {'d02': 3, 'd13': 2}
}

# Cargar modelo base en español sin afinar
modelo_sbert = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')

def cos_sim_np(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

def ndcg_medio(model_instance):
    scores_ndcg = []
    # Generar embeddings de todos los documentos crudos del corpus
    doc_ids_list = list(crudo.keys())
    doc_texts_list = [crudo[did] for did in doc_ids_list]
    emb_docs = model_instance.encode(doc_texts_list, show_progress_bar=False)
    emb_dict = {did: emb_docs[idx] for idx, did in enumerate(doc_ids_list)}
    
    for qid, relevancias in qrels.items():
        q_emb = model_instance.encode(qid, show_progress_bar=False)
        scores = []
        for did in doc_ids_list:
            sim = cos_sim_np(q_emb, emb_dict[did])
            scores.append((did, sim))
        ranking = [item[0] for item in sorted(scores, key=lambda x: x[1], reverse=True)]
        
        # Calcular nDCG@5
        top_k = ranking[:5]
        dcg = 0.0
        for idx, doc in enumerate(top_k):
            r = relevancias.get(doc, 0)
            dcg += (2**r - 1) / math.log2(idx + 2)
        ganancias_ideales = sorted([g for g in relevancias.values()], reverse=True)[:5]
        idcg = sum((2**g - 1) / math.log2(idx + 2) for idx, g in enumerate(ganancias_ideales))
        scores_ndcg.append(dcg / idcg if idcg > 0 else 0.0)
        
    return np.mean(scores_ndcg)

ndcg_base = ndcg_medio(modelo_sbert)
print(f"nDCG@5 Medio de Sentence-BERT base (sin afinar): {ndcg_base:.4f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7825.78it/s]


nDCG@5 Medio de Sentence-BERT base (sin afinar): 0.8818


**A.2** Pares (consulta, documento relevante) desde qrels + fine-tuning con `MultipleNegativesRankingLoss`.

In [15]:
# 1. Generar pares de entrenamiento (consulta, documento positivo) desde qrels (g >= 2)
train_examples = []
for consulta, docs_rel in qrels.items():
    for doc_id, ganancia in docs_rel.items():
        if ganancia >= 2:  # Consideramos pares altamente relevantes
            train_examples.append(InputExample(texts=[consulta, crudo[doc_id]]))

# Crear el DataLoader y la función de pérdida por contraste
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)
train_loss = losses.MultipleNegativesRankingLoss(model=modelo_sbert)

# Ajustar hiperparámetros ligeros para una demostración controlada en Colab
modelo_sbert.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=1,
    show_progress_bar=True
)

ndcg_afinado = ndcg_medio(modelo_sbert)
print(f"\nREPORTE DE COMPARACIÓN DE BUSCADORES:")
print(f"  - FastText (Lab 5):                 ~0.7850 (Léxico denso estático)")
print(f"  - Sentence-BERT base (Sin afinar): {ndcg_base:.4f}")
print(f"  - Sentence-BERT Afinado (Lab 6):   {ndcg_afinado:.4f}")

c:\Users\jhona\Documents\visual studio code\mineria\entorno_pruebas\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss



REPORTE DE COMPARACIÓN DE BUSCADORES:
  - FastText (Lab 5):                 ~0.7850 (Léxico denso estático)
  - Sentence-BERT base (Sin afinar): 0.8818
  - Sentence-BERT Afinado (Lab 6):   0.9172


_Reporten los tres nDCG (FastText, SBERT base, SBERT afinado) y comenten:_ 

**A.3 · Uso del modelo afinado.** Busquen con una consulta nueva usando el encoder ya entrenado.

In [16]:
# Probar el nuevo encoder entrenado con una consulta inédita fuera de qrels
consulta_nueva = "escasez de recursos liquidos e inundacion urbana"
print(f"Búsqueda con modelo SBERT afinado para la consulta: '{consulta_nueva}'")

q_emb_n = modelo_sbert.encode(consulta_nueva, show_progress_bar=False)
resultados_nuevos = []
for did, texto_d in crudo.items():
    d_emb_n = modelo_sbert.encode(texto_d, show_progress_bar=False)
    sim = cos_sim_np(q_emb_n, d_emb_n)
    resultados_nuevos.append((did, titulos[did], sim))

for rank, res in enumerate(sorted(resultados_nuevos, key=lambda x: x[2], reverse=True)[:3]):
    print(f"  Top-{rank+1}: [{res[0]}] (Similitud: {res[2]:.4f}) -> {res[1]}")

# PASO OBLIGATORIO: Liberar VRAM de la Parte A
del modelo_sbert, train_dataloader, train_loss
liberar_memoria()

Búsqueda con modelo SBERT afinado para la consulta: 'escasez de recursos liquidos e inundacion urbana'
  Top-1: [d13] (Similitud: 0.5435) -> Restablecen servicio de agua potable
  Top-2: [d02] (Similitud: 0.4049) -> Crisis hidrica golpea la region
  Top-3: [d01] (Similitud: 0.3475) -> Lluvias provocan inundaciones en Tuxtla


**Liberar memoria** antes del siguiente entrenamiento (clave en T4).

---
## Parte B · Clasificación de sentimiento (dataset real en español)

Entrenamos con **`cardiffnlp/tweet_sentiment_multilingual`** (config `spanish`): miles de ejemplos etiquetados (negativo / neutral / positivo) con splits oficiales train/validation/test. *Si el id cambiara en HF, es la única línea a ajustar; alternativas: `muchocine` (reseñas de cine, 5 clases).*


**B.1** Cargar el dataset y (en T4) submuestrear train para que entrene en minutos.

In [17]:
from datasets import load_dataset

# 1. Cargar directamente los archivos JSONL seguros desde Hugging Face
ds_sentimiento = load_dataset(
    "json", 
    data_files={
        "train": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/spanish/train.jsonl",
        "validation": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/spanish/validation.jsonl",
        "test": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/spanish/test.jsonl"
    }
)

# 2. Configurar el mapeo de clases manual
nombres_clases = ["negative", "neutral", "positive"]
id2lab = {str(i): name for i, name in enumerate(nombres_clases)}
lab2id = {name: i for i, name in enumerate(nombres_clases)}

# Convertir las etiquetas de texto/string a enteros
ds_sentimiento = ds_sentimiento.map(lambda x: {"label": int(x["label"])})

# 3. Submuestreo dinámico adaptado al tamaño real de este split (1839 ejemplos)
tam_train = len(ds_sentimiento['train'])
tam_sub_train = min(1500, tam_train) # Tomamos 1500 para dejarlo redondo y rápido para la T4

ds_train_sub = ds_sentimiento['train'].shuffle(seed=42).select(range(tam_sub_train))
ds_test_sub = ds_sentimiento['test'].shuffle(seed=42).select(range(400))

print(f"Clases del dataset configuradas: {nombres_clases}")
print(f"Tamaño de submuestras -> Train real: {len(ds_train_sub)}, Test: {len(ds_test_sub)}")

Clases del dataset configuradas: ['negative', 'neutral', 'positive']
Tamaño de submuestras -> Train real: 1500, Test: 400


**B.2** Tokenizar con el tokenizer de BETO.

In [18]:
from transformers import AutoTokenizer

CKPT_BETO = 'dccuchile/bert-base-spanish-wwm-cased'
tokenizer_beto = AutoTokenizer.from_pretrained(CKPT_BETO)

def preprocess_function_classification(examples):
    # Padding y truncamiento dinámico uniforme a 128 tokens
    return tokenizer_beto(examples['text'], truncation=True, padding='max_length', max_length=128)

ds_tr_tok = ds_train_sub.map(preprocess_function_classification, batched=True)
ds_te_tok = ds_test_sub.map(preprocess_function_classification, batched=True)

Map: 100%|██████████| 400/400 [00:00<00:00, 5795.54 examples/s]


**B.3** Fine-tuning con `Trainer` (LR 2e-5, pocas épocas, `fp16` para la T4).

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Forzar que los mapeos usen ENTEROS en los IDs para evitar el KeyError en el pipeline
id2lab_corregido = {int(k): v for k, v in id2lab.items()}
lab2id_corregido = {k: int(v) for k, v in lab2id.items()}

# 2. Instanciar BETO con los mapeos corregidos
model_sentimiento = AutoModelForSequenceClassification.from_pretrained(
    CKPT_BETO, 
    num_labels=len(nombres_clases),
    id2label=id2lab_corregido,  # <- Usamos el mapeo con llaves enteras
    label2id=lab2id_corregido
)

def compute_metrics_classification(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {"accuracy": acc, "f1_macro": f1}

training_args_cls = TrainingArguments(
    output_dir="./results_sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",  
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True,  
    logging_steps=50,
    report_to="none"
)

trainer_cls = Trainer(
    model=model_sentimiento,
    args=training_args_cls,
    train_dataset=ds_tr_tok,
    eval_dataset=ds_te_tok,
    compute_metrics=compute_metrics_classification,
)

# 3. Re-entrenar con la configuración corregida
trainer_cls.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24635.60it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from differe

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.034725,0.841400,0.597500,0.575810
2,0.654144,0.809475,0.647500,0.648385


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]
c:\Users\jhona\Documents\visual studio code\mineria\entorno_pruebas\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.10it/s]


TrainOutput(global_step=188, training_loss=0.7956312869457488, metrics={'train_runtime': 898.3344, 'train_samples_per_second': 3.34, 'train_steps_per_second': 0.209, 'total_flos': 197335063296000.0, 'train_loss': 0.7956312869457488, 'epoch': 2.0})

In [25]:
# Corre esta celda para arreglar la configuración sin reentrenar
model_sentimiento.config.id2label = {int(k): v for k, v in id2lab.items()}
model_sentimiento.config.label2id = {k: int(v) for k, v in lab2id.items()}

print("Mapeos corregidos en memoria:", model_sentimiento.config.id2label)

Mapeos corregidos en memoria: {0: 'negative', 1: 'neutral', 2: 'positive'}


**B.4** Análisis de errores: matriz de confusión y reporte por clase.

In [26]:
# Obtener predicciones sobre el set de prueba
predictions_output = trainer_cls.predict(ds_te_tok)
y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = ds_te_tok['label']

print("=== MATRIZ DE CONFUSIÓN ===")
print(confusion_matrix(y_true, y_pred))
print("\n=== REPORTE DE CLASIFICACIÓN POR CLASE ===")
print(classification_report(y_true, y_pred, target_names=nombres_clases))

c:\Users\jhona\Documents\visual studio code\mineria\entorno_pruebas\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


=== MATRIZ DE CONFUSIÓN ===
[[91 23  6]
 [48 74 22]
 [15 27 94]]

=== REPORTE DE CLASIFICACIÓN POR CLASE ===
              precision    recall  f1-score   support

    negative       0.59      0.76      0.66       120
     neutral       0.60      0.51      0.55       144
    positive       0.77      0.69      0.73       136

    accuracy                           0.65       400
   macro avg       0.65      0.65      0.65       400
weighted avg       0.65      0.65      0.65       400



_¿Qué clase es la más difícil? ¿Accuracy o F1-macro es mejor criterio aquí? ¿Por qué?_ La clase más difícil de predecir es consistentemente la etiqueta neutral. Esto ocurre porque los textos neutrales carecen de marcadores léxicos fuertemente polarizados (emoticonos, adjetivos intensos) y se sitúan en una zona de frontera semántica difusa, confundiéndose fácilmente tanto con estructuras positivas como negativas. El F1-macro es un criterio infinitamente superior al Accuracy en esta tarea. Los conjuntos de datos reales de redes sociales sufren de un severo desbalance de clases (muchos más tuits neutrales o negativos que positivos). Mientras que el Accuracy puede inflarse artificialmente prediciendo bien únicamente la clase mayoritaria, el F1-macro penaliza con rigor el rendimiento deficiente en las categorías minoritarias al promediar los scores F1 de cada clase de forma equitativa.

**B.5 · Uso del modelo afinado** — transferencia al corpus chiapaneco. El modelo se entrenó con tuits; veamos qué predice sobre noticias.

In [27]:
from transformers import pipeline

# Construir pipeline de inferencia nativo con el checkpoint optimizado
pipeline_sentimiento = pipeline('text-classification', model=model_sentimiento, tokenizer=tokenizer_beto, device=0)

frases_prueba = [
    "La devastadora sequía arruinó por completo todas las hectáreas de maíz y café.",
    "El gobernador inauguró un moderno centro tecnológico de inteligencia artificial.",
    "Avanza la pavimentación de la carretera estatal de Tuxtla Gutiérrez."
]

print("=== EVALUACIÓN DE DOMAIN SHIFT EN EL CORPUS ===")
for f in frases_prueba:
    res = pipeline_sentimiento(f)[0]
    print(f"Texto: '{f}' -> Predicción: {res['label']} (Score: {res['score']:.4f})")

# PASO OBLIGATORIO: Liberar VRAM de la Parte B
del model_sentimiento, trainer_cls, pipeline_sentimiento
liberar_memoria()

=== EVALUACIÓN DE DOMAIN SHIFT EN EL CORPUS ===
Texto: 'La devastadora sequía arruinó por completo todas las hectáreas de maíz y café.' -> Predicción: negative (Score: 0.7322)
Texto: 'El gobernador inauguró un moderno centro tecnológico de inteligencia artificial.' -> Predicción: neutral (Score: 0.5286)
Texto: 'Avanza la pavimentación de la carretera estatal de Tuxtla Gutiérrez.' -> Predicción: neutral (Score: 0.5626)


**Liberar memoria** antes de la Parte C.

---
## Parte C · NER con CoNLL-2002 (español)

Entrenamos NER con **`conll2002`** config `es`, el estándar en español: esquema BIO con PER/ORG/LOC/MISC y miles de oraciones anotadas. *Si la carga falla por la versión de `datasets`, prueben `load_dataset('conll2002','es', trust_remote_code=True)` o el espejo `eriktks/conll2002`.*


**C.1** Cargar el dataset y leer el esquema de etiquetas desde sus features.

In [32]:
import pandas as pd
from datasets import Dataset, DatasetDict

print("Descargando CoNLL-2002 via Pandas de forma segura...")

# 1. Forzar la descarga directa usando HTTP plano con Pandas (esquivando el parser de HF)
df_train = pd.read_parquet("https://huggingface.co/datasets/conll2002/resolve/refs%2Fconvert%2Fparquet/es/train/0000.parquet")
df_val = pd.read_parquet("https://huggingface.co/datasets/conll2002/resolve/refs%2Fconvert%2Fparquet/es/validation/0000.parquet")
df_test = pd.read_parquet("https://huggingface.co/datasets/conll2002/resolve/refs%2Fconvert%2Fparquet/es/test/0000.parquet")

# 2. Reconstruir el DatasetDict nativo de Hugging Face a partir de los DataFrames
ds_ner = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "validation": Dataset.from_pandas(df_val),
    "test": Dataset.from_pandas(df_test)
})

# 3. Definir el esquema oficial del estándar CoNLL-2002 en Español
etiquetas_ner = [
    "O",          # Outside
    "B-PER",      # Inicio Persona
    "I-PER",      # Continuación Persona
    "B-ORG",      # Inicio Organización
    "I-ORG",      # Continuación Organización
    "B-LOC",      # Inicio Lugar
    "I-LOC",      # Continuación Lugar
    "B-MISC",     # Inicio Miscelánea
    "I-MISC"      # Continuación Miscelánea
]

# 4. Mapeos con enteros para asegurar compatibilidad con la cabeza de clasificación
id2lab_ner = {i: name for i, name in enumerate(etiquetas_ner)}
lab2id_ner = {name: i for i, name in enumerate(etiquetas_ner)}

print(f"\n¡Logrado! Dataset CoNLL-2002 cargado en memoria exitosamente.")
print(f"Estructura del set de entrenamiento: {ds_ner['train']}")

Descargando CoNLL-2002 via Pandas de forma segura...

¡Logrado! Dataset CoNLL-2002 cargado en memoria exitosamente.
Estructura del set de entrenamiento: Dataset({
    features: ['id', 'tokens', 'pos_tags', 'ner_tags'],
    num_rows: 8324
})


**C.2 — el corazón del lab.** Tokenizar y **alinear etiquetas con subpalabras**: la etiqueta va a la **primera** subpalabra de cada palabra; las demás (y `[CLS]`/`[SEP]`) se marcan con `-100` para ignorarlas en la pérdida.

In [33]:
def tokeniza_y_alinea_subpalabras(examples):
    # Procesar secuencias pre-segmentadas en palabras
    tokenized_inputs = tokenizer_beto(examples['tokens'], truncation=True, is_split_into_words=True, max_length=128)
    
    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Tokens especiales de BERT ([CLS], [SEP]) se marcan con -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Es la PRIMERA subpalabra de un token: se le asigna la etiqueta original
                label_ids.append(label[word_idx])
            else:
                # Es una subpalabra intermedia resultante de la fragmentación (WordPiece): se ignora (-100)
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

# Mapear transformaciones vectoriales al dataset completo
ds_ner_tok = ds_ner.map(tokeniza_y_alinea_subpalabras, batched=True, remove_columns=ds_ner['train'].column_names)

Map: 100%|██████████| 1518/1518 [00:00<00:00, 11949.45 examples/s]


**C.3** Fine-tuning con `AutoModelForTokenClassification` (`fp16`, T4).

In [35]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments

# 1. Instanciar el modelo agregando una cabeza de clasificación por token
model_ner = AutoModelForTokenClassification.from_pretrained(
    CKPT_BETO,
    num_labels=len(etiquetas_ner),
    id2label=id2lab_ner,
    label2id=lab2id_ner
)

# 2. El colector de datos se encarga de aplicar padding dinámico tanto a entradas como a las etiquetas de control (-100)
data_collator_ner = DataCollatorForTokenClassification(tokenizer=tokenizer_beto)

# 3. Configurar los argumentos de entrenamiento con la sintaxis correcta (eval_strategy)
training_args_ner = TrainingArguments(
    output_dir="./results_ner",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",  # <- CORREGIDO: Se cambió 'evaluation_strategy' por 'eval_strategy'
    save_strategy="epoch",
    fp16=True,
    logging_steps=50,
    report_to="none"
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 28041.74it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing

**C.4** Evaluación con **seqeval** (a nivel de entidad, no de token).

In [37]:
from seqeval.metrics import f1_score as seq_f1
from transformers import Trainer

def compute_metrics_ner(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    # Reconstruir las secuencias reales ignorando el índice de máscara -100
    true_predictions = [
        [etiquetas_ner[p_idx] for (p_idx, l_idx) in zip(prediction, label) if l_idx != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [etiquetas_ner[l_idx] for (p_idx, l_idx) in zip(prediction, label) if l_idx != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    return {"f1": seq_f1(true_labels, true_predictions)}

trainer_ner = Trainer(
    model=model_ner,
    args=training_args_ner,
    train_dataset=ds_ner_tok['train'].select(range(2000)), 
    eval_dataset=ds_ner_tok['validation'].select(range(400)),
    data_collator=data_collator_ner,
    processing_class=tokenizer_beto, # <- CORREGIDO: Se cambió 'tokenizer' por 'processing_class'
    compute_metrics=compute_metrics_ner
)

trainer_ner.train()

c:\Users\jhona\Documents\visual studio code\mineria\entorno_pruebas\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1
1,0.122475,0.144846,0.807512
2,0.053872,0.143340,0.814765


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.10it/s]
c:\Users\jhona\Documents\visual studio code\mineria\entorno_pruebas\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.81it/s]


TrainOutput(global_step=250, training_loss=0.1720079107284546, metrics={'train_runtime': 777.4595, 'train_samples_per_second': 5.145, 'train_steps_per_second': 0.322, 'total_flos': 163222815835008.0, 'train_loss': 0.1720079107284546, 'epoch': 2.0})

_¿Por qué seqeval y no accuracy por token? ¿Qué tipo de entidad cuesta más?_ No se utiliza Accuracy por token debido al sesgo masivo de la clase mayoritaria O (Outside). En cualquier texto, más del 90% de los tokens no son entidades. Un modelo perezoso que prediga siempre O obtendría un Accuracy superior al 90%, siendo completamente inservible. seqeval evalúa a nivel de entidad completa exigiendo la concordancia exacta de la estructura sintáctica BIO (por ejemplo, que un bloque B-ORG I-ORG comience y cierre de forma idéntica). La entidad que típicamente cuesta más extraer es MISC (Miscelánea), debido a su enorme varianza estructural (puede agrupar títulos de películas, eventos, leyes o premios), careciendo de patrones capitalizados consistentes como los nombres propios de personas (PER) o locaciones (LOC).

**C.5 · Uso del modelo afinado** — cierre del círculo: extraer entidades del **corpus chiapaneco**.

In [38]:
# Instanciar pipeline especializado en NER agregando las subpalabras
pipeline_ner = pipeline('ner', model=model_ner, tokenizer=tokenizer_beto, aggregation_strategy='simple', device=0)

# Aplicar inferencia directa sobre el texto real crudo de una de nuestras noticias
texto_chiapas = crudo['d01'] # "Lluvias provocan inundaciones en Tuxtla..."
print(f"Analizando documento d01: '{texto_chiapas[:90]}...'")

entidades_extraidas = pipeline_ner(texto_chiapas)
print("\nEntidades detectadas por el modelo BETO Afinado:")
for ent in entidades_extraidas:
    print(f"  - Entidad: {ent['word']:<20} | Tipo: {ent['entity_group']:<5} | Score: {ent['score']:.4f}")

# Cierre final de recursos
del model_ner, trainer_ner, pipeline_ner
liberar_memoria()

Analizando documento d01: 'Las  fuertes lluvias   provocaron inundaciones en varias colonias del sur de Tuxtla Gutier...'

Entidades detectadas por el modelo BETO Afinado:
  - Entidad: Tu                   | Tipo: LOC   | Score: 0.9815
  - Entidad: ##xtla               | Tipo: LOC   | Score: 0.7223
  - Entidad: Gutierrez 😟          | Tipo: LOC   | Score: 0.6574
  - Entidad: Prote                | Tipo: ORG   | Score: 0.9409
  - Entidad: ##ccion Civil        | Tipo: ORG   | Score: 0.8335
  - Entidad: chiapasparalelo      | Tipo: ORG   | Score: 0.4685


**Liberar memoria** al terminar.

---
## Síntesis

Las tres partes usaron **el mismo BERT preentrenado** y solo cambiaron la cabeza y los datos: cabeza siamesa con sus qrels (A), `[CLS]` + lineal con un dataset de sentimiento real (B), y una etiqueta por token con CoNLL-2002 (C). Cada modelo afinado se aplicó **sobre su propio corpus**, y entre entrenamientos se liberó memoria para sostener la sesión en una T4. Ese paradigma —preentrenar una vez, adaptar barato— es la base de los sistemas RAG de la Unidad 3.


## Entregables — Lab 6
- [ ] **A:** fine-tuning de Sentence-BERT + los tres nDCG + búsqueda con el modelo afinado.
- [ ] **B:** clasificación con dataset real + matriz de confusión + inferencia sobre el corpus.
- [ ] **C:** NER con CoNLL-2002 + alineación de subpalabras + seqeval + extracción sobre el corpus.
- [ ] Celdas de liberación de memoria ejecutadas entre partes.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
